# 12장. 작은 GPT 조립하기

임베딩 → 블록 6층 → 출력층을 이어 작은 GPT를 조립하고, 손실로 얼마나 틀리는지 재고, 파라미터 수를 세고, 학습 전 GPT로 글을 만들어 봅니다.

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 구조 참고: [nanoGPT](https://github.com/karpathy/nanoGPT)

## 실습 12-1 준비

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import math
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, load_stories, encode_texts, get_batch

torch.manual_seed(42)
vocab = Vocab.build()
stories = load_stories(200)
data = encode_texts(stories, vocab)
print("글자 수:", len(data))

## 실습 12-2 모델 크기 정하기: 설정

In [ ]:
from modu_llm import GPTConfig

cfg = GPTConfig()
print("사전 크기:", cfg.vocab_size)
print("문맥 길이:", cfg.block_size)
print("벡터 길이:", cfg.n_embd)
print("헤드 수:", cfg.n_head)
print("블록 층 수:", cfg.n_layer)
print("글자 사전과 크기가 같은가요?", cfg.vocab_size == len(vocab))

## 실습 12-3 부품 이어 붙이기

In [ ]:
from modu_llm import Embeddings, Block

class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = Embeddings(cfg)                          # ① 글자 카드 + 자리 카드
        self.blocks = nn.ModuleList()
        for i in range(cfg.n_layer):
            self.blocks.append(Block(cfg))                  # ② 블록 6층
        self.ln = nn.LayerNorm(cfg.n_embd)                  # ③ 마지막 정리
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size)   # ④ 출력층

    def forward(self, idx, targets=None):
        x = self.emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln(x)
        logits = self.head(x)
        if targets is None:
            return logits, None
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

model = TinyGPT(cfg)
print(model.head)

## 실습 12-4 모델이 얼마나 틀리는지 재기

In [ ]:
x, y = get_batch(data, cfg.block_size, 4)
logits, loss = model(x, y)
print("x 모양:", x.shape)
print("logits 모양:", logits.shape)
print(f"학습 전 손실: {loss.item():.2f}")
print(f"ln(사전 크기): {math.log(len(vocab)):.2f}")

## 실습 12-5 손실을 손으로 확인하기

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0]])   # 글자 3개의 점수
answer = torch.tensor([0])                  # 정답은 0번 글자
probs = F.softmax(scores, dim=-1)
print("확률:", probs.round(decimals=3))
print(f"-ln(정답 확률): {-math.log(probs[0, 0].item()):.4f}")
print(f"cross_entropy: {F.cross_entropy(scores, answer).item():.4f}")

## 실습 12-6 파라미터 수 세기

In [ ]:
def count(m):
    total = 0
    for p in m.parameters():
        total = total + p.numel()
    return total

print(f"임베딩: {count(model.emb):,}")
print(f"블록 6층: {count(model.blocks):,}")
print(f"출력층: {count(model.ln) + count(model.head):,}")
print(f"전체: {count(model):,}")

## 실습 12-7 학습 전 GPT로 글 만들기

In [ ]:
from modu_llm import generate

text = generate(model, vocab, "어느 날, ", max_new_tokens=80, stop_at_end=False)
print("어느 날, " + text)

## 실습 12-8 더 해 보기: 크기를 바꾸면

In [ ]:
for n_layer, n_embd in [(6, 256), (12, 256), (6, 512)]:
    m = TinyGPT(GPTConfig(n_layer=n_layer, n_embd=n_embd))
    print(f"층 {n_layer}, 벡터 길이 {n_embd}: {count(m):,}")